# Austrian Tax Law LLM - Phase 2: Apply Models to Dataset

## Overview

This notebook runs three language model approaches on 643 Austrian tax law questions
and saves the results as separate CSV files for submission.

**Three approaches:**
1. Inference only - Llama 3.3 70B via Groq API (the one permitted external API)
2. Fine-tuning - Llama 3.1 8B with QLoRA, trained on external data
3. RAG - retrieve relevant legal text passages, then generate answers with the local model

**Input:**
- `dataset_clean.csv` - 643 test questions (id, prompt). Used only as test input, never for training.
- `ft_training_data.jsonl` - external training examples for fine-tuning
- `data/legal_docs/*.txt` - legal text files for RAG retrieval

**Output:**
- `inference_submission.csv` - answers from the Groq API model
- `ft_submission.csv` - answers from the fine-tuned local model
- `rag_submission.csv` - answers from RAG (retrieval + local model)

Each output file has two columns: `id` and `answer`.

**Runtime strategy:**
Groq API calls run in a background thread (~21 min for 643 questions at free-tier rate limits).
Fine-tuning and local inference run on the GPU in the foreground at the same time.
Total wall time is roughly the slower of the two, not the sum.

**API rule:** Only one approach uses an external API (Groq for inference).
Fine-tuning and RAG both run locally on the Colab T4 GPU.

## Step 1: Install Required Libraries

Unsloth handles efficient QLoRA fine-tuning on a T4 GPU.
Groq is the API client. sentence-transformers and faiss-cpu are for RAG retrieval.

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q groq sentence-transformers faiss-cpu trl datasets
!pip install -q --no-deps peft accelerate bitsandbytes xformers

## Step 2: Load API Keys and Test Data

API keys are stored in Colab Secrets (Settings > Secrets), not hardcoded.
- `GROQ_API_KEY` from https://console.groq.com
- `HF_TOKEN` from https://huggingface.co/settings/tokens

In [ ]:
import os
import time
import pandas as pd

# Load API keys from Colab Secrets
try:
    from google.colab import userdata
    groq_api_key = userdata.get("GROQ_API_KEY")
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    groq_api_key = os.getenv("GROQ_API_KEY", "")
    hf_token = os.getenv("HF_TOKEN", "")

if not groq_api_key:
    print("WARNING: GROQ_API_KEY is not set.")
else:
    print("Groq API key loaded.")

if not hf_token:
    print("WARNING: HF_TOKEN is not set.")
else:
    print("Hugging Face token loaded.")

# Load the test dataset
df = pd.read_csv("/content/dataset_clean.csv")
print(f"Loaded {len(df)} test questions.")
print(df.head(3).to_string())

## Step 3: Groq API Inference (Background)

This is the only approach that uses an external API.

We validate the key with one test call. If it works, we start a background thread
that sends all 643 questions to the Groq API one by one. The thread runs while
the GPU handles fine-tuning and RAG in the foreground.

Rate limit handling:
- 1.5 second pause between calls (safe for free-tier ~30 req/min)
- Up to 3 retries per question with increasing wait times on rate limit errors
- If 5 questions in a row fail, the thread stops early and marks remaining questions as errors

In [ ]:
from groq import Groq
import threading

client = Groq(api_key=groq_api_key)

# System prompt: answer Austrian tax law questions in German, cite legal basis
SYSTEM_MSG_INFERENCE = (
    "Du bist ein Experte fuer oesterreichisches Steuerrecht. "
    "Beantworte die folgende Frage kurz, praezise und auf Deutsch. "
    "Nenne die relevante Gesetzesgrundlage, wenn moeglich."
)

def call_groq(question, system_msg=SYSTEM_MSG_INFERENCE, max_retries=3):
    """Send one question to the Groq API. Returns the answer text or an error string."""
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "system", "content": system_msg},
                    {"role": "user", "content": question}
                ],
                max_tokens=300,
                temperature=0.1
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            error_str = str(e).lower()
            if "rate_limit" in error_str or "429" in error_str:
                wait_time = 10 * (attempt + 1)
                time.sleep(wait_time)
            else:
                return f"ERROR: {e}"
    return "ERROR: rate limit exceeded after retries"

# Validate API key with one test question
print("Testing Groq API key...")
test_answer = call_groq("Was ist die Koerperschaftsteuer in Oesterreich?")
api_valid = not test_answer.startswith("ERROR")

if not api_valid:
    print(f"API test failed: {test_answer}")
else:
    print(f"API works. Test answer preview: {test_answer[:100]}...")

# Shared list for results. The background thread fills this in.
inference_answers = [None] * len(df)
inference_done = threading.Event()

def run_inference_background():
    """Run all Groq API calls sequentially in a background thread."""
    questions = df["prompt"].tolist()
    consecutive_errors = 0
    start = time.time()

    for i, question in enumerate(questions):
        answer = call_groq(question)
        inference_answers[i] = answer

        # Track consecutive errors for early stopping
        if answer.startswith("ERROR"):
            consecutive_errors += 1
        else:
            consecutive_errors = 0

        # If 5 questions in a row fail, stop early
        if consecutive_errors >= 5:
            print(f"\n[Background] 5 consecutive errors at question {i+1}. Stopping early.")
            for j in range(i + 1, len(questions)):
                inference_answers[j] = "ERROR: stopped early after repeated failures"
            break

        # 1.5s pause between calls
        time.sleep(1.5)

        # Progress update every 50 questions
        if (i + 1) % 50 == 0:
            elapsed = time.time() - start
            print(f"[Background] Inference progress: {i+1}/{len(questions)} ({elapsed:.0f}s)")

    elapsed = time.time() - start
    errors = sum(1 for a in inference_answers if a and a.startswith("ERROR"))
    print(f"\n[Background] Inference complete: {elapsed:.0f}s ({elapsed/60:.1f} min), errors: {errors}")
    inference_done.set()

# Start background thread
if api_valid:
    inference_thread = threading.Thread(target=run_inference_background, daemon=True)
    inference_thread.start()
    print("Groq inference started in background. Continuing with fine-tuning...")
else:
    # Fill all with error markers
    inference_answers = ["ERROR: API key not valid"] * len(df)
    inference_done.set()
    print("Skipping inference (API key issue). Continuing with fine-tuning...")

---
## Approach 2: Fine-Tuned Model (QLoRA on Llama 3.1 8B)

While the Groq API calls run in the background, we load and fine-tune a local model.

Training data comes from `ft_training_data.jsonl`, which is an external file with
domain-specific Austrian tax law question-answer pairs. This file is not derived from
`dataset_clean.csv`, so there is no data leakage.

QLoRA means: the base model weights are frozen in 4-bit precision, and we only train
small LoRA adapter layers on top. This fits on a T4 GPU (16 GB VRAM).

In [ ]:
import json

FT_TRAINING_FILE = "/content/ft_training_data.jsonl"
ft_data_available = os.path.exists(FT_TRAINING_FILE)

if ft_data_available:
    ft_examples = []
    with open(FT_TRAINING_FILE, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                ft_examples.append(json.loads(line))
    print(f"Fine-tuning training data: {len(ft_examples)} examples loaded.")
    # Show one example so we can verify the format
    print(f"Example keys: {list(ft_examples[0].keys())}")
else:
    print(f"Training file not found at {FT_TRAINING_FILE}.")
    print("Fine-tuning will be skipped. ft_submission.csv will contain placeholders.")

In [ ]:
if not ft_data_available:
    print("Skipping model load (no training data).")
else:
    from huggingface_hub import login
    login(token=hf_token)

    from unsloth import FastLanguageModel

    # Load Llama 3.1 8B in 4-bit quantization
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
        max_seq_length=768,
        load_in_4bit=True
    )

    # Add LoRA adapter layers for training
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        lora_alpha=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        use_gradient_checkpointing="unsloth"
    )

    print("Model loaded and LoRA adapters added.")

In [ ]:
if not ft_data_available:
    print("Skipping.")
else:
    from datasets import Dataset

    # System message used during training and later during inference
    FT_SYSTEM_MSG = (
        "Du bist ein Experte fuer oesterreichisches Steuerrecht. "
        "Beantworte die Frage praezise und nenne die relevanten Rechtsgrundlagen."
    )

    def build_chat_text(system_msg, user_msg, assistant_msg=None):
        """Build a Llama 3.1 chat-formatted string.

        This follows the exact template the model was pretrained with.
        During training, we include the assistant response.
        During inference, we leave it open so the model generates the answer.
        """
        text = (
            "<|begin_of_text|>"
            "<|start_header_id|>system<|end_header_id|>\n\n"
            f"{system_msg}<|eot_id|>"
            "<|start_header_id|>user<|end_header_id|>\n\n"
            f"{user_msg}<|eot_id|>"
            "<|start_header_id|>assistant<|end_header_id|>\n\n"
        )
        if assistant_msg is not None:
            text += f"{assistant_msg}<|eot_id|>"
        return text

    def format_example(example):
        return {"text": build_chat_text(FT_SYSTEM_MSG, example["prompt"], example["answer"])}

    formatted_dataset = Dataset.from_list(ft_examples).map(format_example)
    print(f"Formatted training dataset: {len(formatted_dataset)} examples.")

In [ ]:
if not ft_data_available:
    print("Skipping training.")
else:
    from trl import SFTTrainer
    from transformers import TrainingArguments

    training_args = TrainingArguments(
        output_dir="/content/ft_model",
        num_train_epochs=1,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,  # effective batch size = 2 * 4 = 8
        learning_rate=2e-4,
        warmup_steps=10,
        fp16=True,
        logging_steps=10,
        save_strategy="no",          # do not save checkpoints (saves disk and time)
        optim="adamw_8bit",          # memory-efficient optimizer
        report_to="none",
        seed=42
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=formatted_dataset,
        dataset_text_field="text",
        max_seq_length=768,
        packing=True,                # pack multiple short examples into one sequence
        args=training_args
    )

    print("Starting fine-tuning...")
    t0 = time.time()
    trainer.train()
    print(f"Fine-tuning complete in {time.time() - t0:.0f}s.")

### Run Fine-Tuned Model on Test Set

We generate answers for all 643 questions using the fine-tuned model in batches of 4.

Technical note: `use_cache=False` is needed during generation because padded batches
cause a shape mismatch in the rotary position embedding cache. This is a known issue
with Unsloth/Llama and the standard workaround.

In [ ]:
if not ft_data_available:
    print("Fine-tuning was skipped. Saving placeholder file.")
    ft_submission = pd.DataFrame({
        "id": df["id"].tolist(),
        "answer": ["PLACEHOLDER: fine-tuning data not available"] * len(df)
    })
    ft_submission.to_csv("/content/ft_submission.csv", index=False)

else:
    import torch
    FastLanguageModel.for_inference(model)

    ft_answers = []
    BATCH_SIZE = 4
    questions = df["prompt"].tolist()

    # Build all prompts (no assistant answer = model generates it)
    prompts = [build_chat_text(FT_SYSTEM_MSG, q) for q in questions]

    print(f"Running FT inference on {len(questions)} questions (batch size {BATCH_SIZE})...")
    t0 = time.time()

    for i in range(0, len(prompts), BATCH_SIZE):
        batch = prompts[i : i + BATCH_SIZE]

        inputs = tokenizer(
            batch, return_tensors="pt", padding=True,
            truncation=True, max_length=512
        ).to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=150,
                temperature=0.1,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
                use_cache=False   # required workaround for rotary embedding error
            )

        # Decode only the newly generated tokens (skip the input)
        input_len = inputs["input_ids"].shape[1]
        for out in outputs:
            ft_answers.append(
                tokenizer.decode(out[input_len:], skip_special_tokens=True).strip()
            )

        done = min(i + BATCH_SIZE, len(prompts))
        if done % 50 == 0 or done == len(prompts):
            print(f"  {done}/{len(prompts)}")

    print(f"FT inference done in {time.time() - t0:.0f}s.")

    ft_submission = pd.DataFrame({"id": df["id"].tolist(), "answer": ft_answers})
    ft_submission.to_csv("/content/ft_submission.csv", index=False)
    print(f"Saved: ft_submission.csv ({len(ft_submission)} rows)")

---
## Approach 3: RAG (Retrieval-Augmented Generation)

RAG retrieves relevant legal text passages and includes them in the prompt before
generating an answer. This gives the model specific legal context it would not
otherwise have access to.

Steps:
1. Load all `.txt` files from `data/legal_docs/` and split them into overlapping chunks
2. Embed all chunks with a multilingual sentence-transformer model
3. Build a FAISS index for fast similarity search
4. For each question, find the top 3 most relevant chunks
5. Generate an answer using the local fine-tuned model with the retrieved chunks as context

The local model is reused from fine-tuning (already loaded in GPU memory).
This avoids a second round of API calls and stays within the one-API rule.

**Limitation:** RAG quality depends on how well the legal document corpus covers the
topics in the test questions. If a topic is not covered in the documents, the retrieved
chunks may not be relevant.

In [ ]:
LEGAL_DOCS_FOLDER = "/content/data/legal_docs"

if os.path.isdir(LEGAL_DOCS_FOLDER):
    doc_files = [f for f in os.listdir(LEGAL_DOCS_FOLDER) if f.endswith(".txt")]
else:
    doc_files = []

rag_docs_available = len(doc_files) > 0

if rag_docs_available:
    print(f"Found {len(doc_files)} legal document files in {LEGAL_DOCS_FOLDER}/")
    for f in sorted(doc_files)[:5]:
        print(f"  - {f}")
    if len(doc_files) > 5:
        print(f"  ... and {len(doc_files) - 5} more")
else:
    print(f"No .txt files found in {LEGAL_DOCS_FOLDER}/.")
    print("RAG will be skipped. rag_submission.csv will contain placeholders.")

In [ ]:
if not rag_docs_available:
    print("Skipping RAG index (no documents).")
else:
    # --- Load and chunk documents ---
    CHUNK_SIZE = 300       # words per chunk
    CHUNK_OVERLAP = 50     # overlapping words between chunks
    all_chunks = []

    for filename in sorted(doc_files):
        filepath = os.path.join(LEGAL_DOCS_FOLDER, filename)
        with open(filepath, "r", encoding="utf-8") as f:
            words = f.read().split()

        # Split into overlapping chunks
        pos = 0
        while pos < len(words):
            chunk_text = " ".join(words[pos : pos + CHUNK_SIZE])
            all_chunks.append(chunk_text)
            pos += CHUNK_SIZE - CHUNK_OVERLAP

    print(f"Created {len(all_chunks)} text chunks from {len(doc_files)} documents.")

    # --- Embed chunks and build FAISS index ---
    import numpy as np
    import faiss
    from sentence_transformers import SentenceTransformer

    # This model handles German text well
    embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

    # Encode all chunks into vectors
    chunk_vecs = embed_model.encode(
        all_chunks, show_progress_bar=True,
        convert_to_numpy=True, batch_size=64
    )

    # Normalize for cosine similarity
    faiss.normalize_L2(chunk_vecs)

    # Build the search index
    index = faiss.IndexFlatIP(chunk_vecs.shape[1])
    index.add(chunk_vecs.astype("float32"))
    print(f"FAISS index built: {index.ntotal} vectors, dimension {chunk_vecs.shape[1]}.")

    # --- Retrieve top-3 chunks for each question ---
    TOP_K = 3
    q_vecs = embed_model.encode(df["prompt"].tolist(), convert_to_numpy=True, batch_size=64)
    faiss.normalize_L2(q_vecs)
    distances, indices_rag = index.search(q_vecs.astype("float32"), k=TOP_K)
    print(f"Retrieved top-{TOP_K} chunks for all {len(df)} questions.")

In [ ]:
if not rag_docs_available:
    print("RAG skipped. Saving placeholder file.")
    rag_submission = pd.DataFrame({
        "id": df["id"].tolist(),
        "answer": ["PLACEHOLDER: legal documents not available"] * len(df)
    })
    rag_submission.to_csv("/content/rag_submission.csv", index=False)

elif not ft_data_available:
    # If the fine-tuned model is not available, we cannot run RAG locally.
    # We do NOT fall back to the Groq API here, because that would mean
    # two of three approaches use an external API, violating the project rule.
    print("RAG skipped: local model not available (fine-tuning was skipped).")
    print("Cannot use Groq API for RAG because that would violate the one-API rule.")
    rag_submission = pd.DataFrame({
        "id": df["id"].tolist(),
        "answer": ["PLACEHOLDER: local model not available for RAG"] * len(df)
    })
    rag_submission.to_csv("/content/rag_submission.csv", index=False)

else:
    import torch

    RAG_SYSTEM_MSG = (
        "Du bist ein Experte fuer oesterreichisches Steuerrecht. "
        "Dir werden relevante Auszuege aus oesterreichischen Steuergesetzen "
        "und eine Frage gegeben. "
        "Beantworte die Frage kurz und praezise auf Deutsch, "
        "gestuetzt auf die gegebenen Texte."
    )

    questions = df["prompt"].tolist()

    # Build RAG prompts: each question gets its top-3 retrieved chunks as context
    rag_prompts = []
    for i, q in enumerate(questions):
        # Concatenate top-k chunks
        context_parts = []
        for rank in range(TOP_K):
            chunk_idx = indices_rag[i][rank]
            context_parts.append(all_chunks[chunk_idx])
        context = "\n\n".join(context_parts)

        user_msg = f"Relevante Gesetzestexte:\n{context}\n\nFrage: {q}"
        rag_prompts.append(build_chat_text(RAG_SYSTEM_MSG, user_msg))

    rag_answers = []
    BATCH_SIZE = 4

    print(f"Running RAG inference on {len(questions)} questions (batch size {BATCH_SIZE})...")
    t0 = time.time()

    for i in range(0, len(rag_prompts), BATCH_SIZE):
        batch = rag_prompts[i : i + BATCH_SIZE]

        inputs = tokenizer(
            batch, return_tensors="pt", padding=True,
            truncation=True, max_length=512
        ).to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=150,
                temperature=0.1,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
                use_cache=False
            )

        input_len = inputs["input_ids"].shape[1]
        for out in outputs:
            rag_answers.append(
                tokenizer.decode(out[input_len:], skip_special_tokens=True).strip()
            )

        done = min(i + BATCH_SIZE, len(rag_prompts))
        if done % 50 == 0 or done == len(rag_prompts):
            print(f"  {done}/{len(rag_prompts)}")

    print(f"RAG inference done in {time.time() - t0:.0f}s.")

    rag_submission = pd.DataFrame({"id": df["id"].tolist(), "answer": rag_answers})
    rag_submission.to_csv("/content/rag_submission.csv", index=False)
    print(f"Saved: rag_submission.csv ({len(rag_submission)} rows)")

---
## Step 4: Collect Groq Inference Results

The background thread has been running Groq API calls this entire time.
Now we wait for it to finish (if it has not already) and save the inference CSV.

In [ ]:
# Wait for the background inference thread to finish
if not inference_done.is_set():
    print("Waiting for Groq inference to finish...")
    while not inference_done.is_set():
        completed = sum(1 for a in inference_answers if a is not None)
        print(f"  Progress: {completed}/{len(df)}", end="\r")
        inference_done.wait(timeout=15)
    print()

# Fill any remaining None values (should not happen, but just in case)
for i in range(len(inference_answers)):
    if inference_answers[i] is None:
        inference_answers[i] = "ERROR: did not complete"

# Save
inference_submission = pd.DataFrame({
    "id": df["id"].tolist(),
    "answer": inference_answers
})
inference_submission.to_csv("/content/inference_submission.csv", index=False)

errors = sum(1 for a in inference_answers if a.startswith("ERROR"))
print(f"Saved: inference_submission.csv ({len(inference_submission)} rows, {errors} errors)")

---
## Step 5: Verify Output Files

Quick check that all three CSV files exist and have the expected number of rows.

In [ ]:
output_files = [
    "/content/inference_submission.csv",
    "/content/ft_submission.csv",
    "/content/rag_submission.csv"
]

all_ok = True
for filepath in output_files:
    if os.path.exists(filepath):
        sub = pd.read_csv(filepath)
        status = "OK" if len(sub) == len(df) else f"WRONG ROW COUNT ({len(sub)})"
        if len(sub) != len(df):
            all_ok = False
        print(f"{os.path.basename(filepath)}: {len(sub)} rows - {status}")
        # Show first row as a sanity check
        print(f"  First answer preview: {str(sub['answer'].iloc[0])[:80]}...")
        print()
    else:
        all_ok = False
        print(f"MISSING: {filepath}\n")

if all_ok:
    print("All files ready for submission.")
else:
    print("Some files have issues. Check above.")

---
## Limitations

**Inference (Groq API):** The Llama 3.3 70B model is a general-purpose model. It has
broad knowledge but is not specialized in Austrian tax law. Answer quality depends on
what the model learned during pretraining.

**Fine-tuning (QLoRA):** We train for only 1 epoch on a small external dataset. The 8B
model is much smaller than the 70B API model, so its general knowledge is more limited.
The quality of answers depends heavily on the quality and coverage of the training data.

**RAG:** Answer quality depends on two things: (1) whether the legal document corpus
contains text relevant to the question, and (2) whether the retrieval step finds the
right passage. With only top-3 retrieval from a limited corpus, some questions may get
irrelevant context. RAG uses the smaller local model for generation.

**General:** None of the three approaches have been evaluated against ground truth answers
in this notebook. Evaluation happens in Phase 3 of the project.

---
## Summary

### What this notebook does

This notebook answers 643 Austrian tax law questions using three different approaches
and saves the results as CSV files.

### How it works step by step

1. We install the required Python libraries.
2. We load the test questions from `dataset_clean.csv`.
3. We start sending questions to the Groq API in a background thread (approach 1: inference).
4. While the API calls are running, we load a smaller language model and fine-tune it
   on external training examples using QLoRA (approach 2: fine-tuning).
5. We run the fine-tuned model on all 643 questions and save the answers.
6. We load legal text documents, split them into chunks, and build a search index.
   For each question, we find the most relevant chunks and include them in the prompt
   before generating an answer with the local model (approach 3: RAG).
7. We wait for the API calls to finish and save those answers too.
8. We verify that all three output files have 643 rows each.

---
## Checklist

### Before running
- Upload `ft_training_data.jsonl` to `/content/`
- Upload legal `.txt` files to `/content/data/legal_docs/`
- Set `GROQ_API_KEY` and `HF_TOKEN` in Colab Secrets (Settings > Secrets)

### After running
- Verify all three CSV files have 643 rows each
- Check for ERROR entries in inference_submission.csv
- Upload CSV files and this notebook to the shared GitHub repository

---
### Note on notebook structure:
This notebook was originally designed to run all three approaches (inference, fine-tuning, RAG) in a single session. During execution, it became clear that each approach requires a fresh GPU runtime on Google Colab's free tier. The fine-tuning section exhausts the available GPU memory, making it impossible to continue with RAG and inference in the same session.
As a result, the submissions were generated as follows:

inference_submission.csv → generated by 01_inference_only.ipynb
ft_submission.csv → generated by this notebook (Austrian_Tax_Law_LLM_Phase2__1_.ipynb), FT section
rag_submission.csv → generated by 03_rag.ipynb


